# 05 — Threshold Tuning, Singleton Handling, Error Analysis (Phases 8, 16-19)

Deeper dive on top of notebook 04's winning model: fine-grained threshold
search, source-specific thresholds, the optional one-parent-per-match
consistency rule and score-gap ambiguity guard, and qualitative error
analysis on the validation split.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import joblib
import pandas as pd
from src import config, features, retrieval, train as train_module, thresholding
from src.data_loader import load_normalized_source, load_ground_truth, load_ground_truth_exploded
from src.inference import VectorizedOtherSide, candidates_for_chunk_and_source
from src.labeling import truth_dict_from_wide
from src.metrics import evaluate_entity_level_f05, pair_level_precision_recall
from src.model import SklearnModelWrapper, RuleBasedModel


In [ ]:
# Load the artifacts written by notebook 04 / src.train
idf_tables = joblib.load(config.MODELS_DIR / "idf_tables.joblib")
name_vec = joblib.load(config.MODELS_DIR / "name_vectorizer.joblib")
addr_vec = joblib.load(config.MODELS_DIR / "addr_vectorizer.joblib")
threshold_info = json.loads((config.MODELS_DIR / "threshold.json").read_text())
model_path = config.MODELS_DIR / "final_model.joblib"
model_wrapper = SklearnModelWrapper.load(model_path) if model_path.exists() else RuleBasedModel()
threshold_info


In [ ]:
s1_norm = load_normalized_source("train", "source1")
s2_norm = load_normalized_source("train", "source2")
s3_norm = load_normalized_source("train", "source3")

all_ids = train_module.sample_source1_ids(s1_norm["entity_id"].tolist(), config.EXPERIMENT_SAMPLE_SIZE, config.RANDOM_SEED)
train_ids, val_ids = train_module.split_source1_ids(all_ids, config.VALIDATION_FRACTION, config.RANDOM_SEED)
truth_full = truth_dict_from_wide(load_ground_truth())
val_truth = {k: v for k, v in truth_full.items() if k in set(val_ids)}

s2_side = VectorizedOtherSide(s2_norm, name_vec, addr_vec)
s3_side = VectorizedOtherSide(s3_norm, name_vec, addr_vec)
val_feat, val_labels, val_recall = train_module.build_feature_table(
    val_ids, s1_norm, s2_side, s3_side, name_vec, addr_vec,
    idf_tables["name_idf"], idf_tables["addr_idf"], load_ground_truth_exploded(), chunk_size=2000,
)
val_scores = val_feat[["entity_id_s1", "entity_id_other"]].assign(score=model_wrapper.predict_proba_pair(val_feat))
val_recall


## Coarse -> fine threshold search on the exact entity-level F0.5 metric

In [ ]:
coarse = thresholding.search_best_threshold(val_scores, val_truth, val_ids)
fine = thresholding.refine_threshold_search(val_scores, val_truth, val_ids, coarse.best_threshold)
fine.table


## Source-specific thresholds (Source1->Source2 vs Source1->Source3)

In [ ]:
source_thresholds = thresholding.search_source_specific_thresholds(val_scores, val_truth, val_ids, fine.best_threshold)
source_thresholds


## Does the one-parent-per-match consistency rule help? (only kept if it measurably improves F0.5)

In [ ]:
no_rule = thresholding.search_best_threshold(val_scores, val_truth, val_ids, apply_consistency_rule=False)
with_rule = thresholding.search_best_threshold(val_scores, val_truth, val_ids, apply_consistency_rule=True)
print("without consistency rule:", no_rule.best_threshold, no_rule.table['macro_f05'].max())
print("with consistency rule:   ", with_rule.best_threshold, with_rule.table['macro_f05'].max())


## Error analysis: false positives / false negatives at the chosen threshold

In [ ]:
final_threshold = fine.best_threshold
preds = thresholding.predictions_at_threshold(val_scores, final_threshold)
result = evaluate_entity_level_f05(preds, val_truth, val_ids)
print(result["macro_f05"], result["macro_precision"], result["macro_recall"], result["singleton_accuracy"])

per_entity = pd.DataFrame([e.__dict__ for e in result["per_entity"]])
worst = per_entity.sort_values("f05").head(20)
worst


In [ ]:
# Inspect a few concrete false-positive / false-negative examples with their raw text,
# to categorize error types (typos, abbreviations, reordering, common names, ...).
s1_lookup = s1_norm.set_index("entity_id")
other_lookup = pd.concat([s2_norm, s3_norm]).set_index("entity_id")

def show_case(s1_id):
    row = s1_lookup.loc[s1_id]
    print("S1:", row["name_raw"], "|", row["address_raw"], "|", row["country"])
    true_ids = val_truth.get(s1_id, set())
    pred_ids = preds.get(s1_id, set())
    for label, ids in [("TRUE", true_ids), ("PRED", pred_ids)]:
        for oid in ids:
            if oid in other_lookup.index:
                r = other_lookup.loc[oid]
                print(f"  [{label}] {oid}: {r['name_raw']} | {r['address_raw']} | {r['country']}")
    print()

for s1_id in worst["source1_id"].head(8):
    show_case(s1_id)


Categorize the printed cases (typo, abbreviation/DBA, address reordering,
missing address component, common business name collision, transliteration,
numeric conflict, country-specific pattern) and, for each recurring
category, propose a targeted fix (a new blocking rule, a new feature, or a
normalization change) -- then re-run notebook 03/04 and confirm the change
actually improves `macro_f05` here before keeping it. Record each iteration
in `experiments/experiment_log.md`.